## Testing DAGGER

### MountainCar

is normalization needed?

In [40]:
import tempfile
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from imitation.algorithms import bc
from imitation.algorithms.dagger import SimpleDAggerTrainer
from stable_baselines3.common.env_util import make_vec_env

from stable_baselines3 import PPO, DDPG, SAC, TD3
from sb3_contrib import TRPO
from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.evaluation import evaluate_policy

PROJECT_ROOT = Path.cwd().parent
#BASELINE_CODE = PROJECT_ROOT / "code" / "baseline_code"
cwd = Path.cwd()
current = cwd
while not (current / "code" / "baseline_code").exists() and current != current.parent:
    current = current.parent

PROJECT_ROOT = current
BASELINE_CODE = PROJECT_ROOT / "code" / "baseline_code"
print("CWD:", cwd)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("BASELINE_CODE:", BASELINE_CODE)

sys.path.insert(0, str(BASELINE_CODE))
print(Path.cwd())
from baseline_enviroments.MountainCar import make_continuous_mountaincar


# ----- Paths -----
TEACHER_DIR = BASELINE_CODE / "baseline_models" / "mountaincar"
DAGGER_OUT_DIR = BASELINE_CODE / "dagger_models" / "mountaincar"
DAGGER_OUT_DIR.mkdir(parents=True, exist_ok=True)

# ----- Map filename prefix -> loader -----
ALGOS = {
    "PPO": PPO,
    #"DDPG": DDPG,
 #   "SAC": SAC,
  #  "TD3": TD3,
   # "TRPO": TRPO,
}

# ----- Shared -----
rng = np.random.default_rng(0)
ENV_FACTORY = make_continuous_mountaincar

# DAgger settings (adjust)
DAGGER_TIMESTEPS = 5_000      # total timesteps for SimpleDAggerTrainer
N_EVAL_EPISODES = 10

results = []

# Find teacher zips like "PPO_mountaincar.zip"
teacher_paths = sorted(TEACHER_DIR.glob("*_mountaincar.zip"))
if not teacher_paths:
    raise FileNotFoundError(f"No teacher models found in: {TEACHER_DIR}")

for teacher_zip in teacher_paths:
    algo_name = teacher_zip.name.split("_")[0]  # "PPO" from "PPO_mountaincar.zip"
    if algo_name not in ALGOS:
        print(f"Skipping unknown algo file: {teacher_zip.name}")
        continue

    Algo = ALGOS[algo_name]
    print(f"\n=== DAgger for {algo_name} teacher: {teacher_zip.name} ===")
    vec_path = TEACHER_DIR / f"{algo_name}_vecnormalize.pkl"

    # Create venv for DAgger
    venv = make_vec_env(ENV_FACTORY, n_envs=1)
    venv = VecNormalize.load(str(vec_path), venv)
    venv.training = False
    venv.norm_reward = False

    # Load teacher; env=None avoids observation-space equality checks
    expert = Algo.load(str(teacher_zip), env=None)

    # BC trainer (student policy)
    bc_trainer = bc.BC(
        observation_space=venv.observation_space,
        action_space=venv.action_space,
        rng=rng,
    )

    with tempfile.TemporaryDirectory(prefix=f"dagger_{algo_name}_") as tmpdir:
        dagger_trainer = SimpleDAggerTrainer(
            venv=venv,
            scratch_dir=tmpdir,
            expert_policy=expert,
            bc_trainer=bc_trainer,
            rng=rng,
        )

        dagger_trainer.train(
            total_timesteps=DAGGER_TIMESTEPS)

        # Evaluate student (must use same obs normalization)
        eval_env = make_vec_env(ENV_FACTORY, n_envs=1)
        eval_env = VecNormalize.load(str(vec_path), eval_env)
        eval_env.training = False
        eval_env.norm_reward = False
        mean_reward, std_reward = evaluate_policy(
            dagger_trainer.policy, eval_env, n_eval_episodes=N_EVAL_EPISODES
        )

        # Save the learned DAgger policy (imitation policies use .save)
        out_path = DAGGER_OUT_DIR / f"DAgger_{algo_name}_mountaincar"
        dagger_trainer.policy.save(str(out_path))
        print(f"Saved DAgger student to: {out_path}")

        results.append(
            {
                "teacher_algo": algo_name,
                "teacher_file": teacher_zip.name,
                "dagger_timesteps": DAGGER_TIMESTEPS,
                "mean_reward": mean_reward,
                "std_reward": std_reward,
                "saved_student": str(out_path),
            }
        )

        eval_env.close()
    venv.close()

df = pd.DataFrame(results).sort_values("mean_reward", ascending=False)
print(df)
df.to_csv(DAGGER_OUT_DIR / "dagger_results.csv", index=False)
print(f"\nWrote results to: {DAGGER_OUT_DIR / 'dagger_results.csv'}")

CWD: /Users/elisalaegsgaard/Desktop/Speciale/GGSpeciale/notebooks
PROJECT_ROOT: /Users/elisalaegsgaard/Desktop/Speciale/GGSpeciale
BASELINE_CODE: /Users/elisalaegsgaard/Desktop/Speciale/GGSpeciale/code/baseline_code
/Users/elisalaegsgaard/Desktop/Speciale/GGSpeciale/notebooks
Skipping unknown algo file: DDPG_mountaincar.zip

=== DAgger for PPO teacher: PPO_mountaincar.zip ===


Saving the dataset (1/1 shards): 100%|██████████| 1/1 [00:00<00:00, 286.54 examples/s]
0batch [00:00, ?batch/s]

--------------------------------
| batch_size        | 32       |
| bc/               |          |
|    batch          | 0        |
|    ent_loss       | -0.00142 |
|    entropy        | 1.42     |
|    epoch          | 0        |
|    l2_loss        | 0        |
|    l2_norm        | 68.5     |
|    loss           | 1.27     |
|    neglogp        | 1.27     |
|    prob_true_act  | 0.284    |
|    samples_so_far | 32       |
| rollout/          |          |
|    return_max     | -49.4    |
|    return_mean    | -50.8    |
|    return_min     | -51.9    |
|    return_std     | 0.965    |
--------------------------------


64batch [00:01, 43.00batch/s]
Saving the dataset (1/1 shards): 100%|██████████| 1/1 [00:00<00:00, 287.93 examples/s]
0batch [00:00, ?batch/s]

--------------------------------
| batch_size        | 32       |
| bc/               |          |
|    batch          | 0        |
|    ent_loss       | -0.00135 |
|    entropy        | 1.35     |
|    epoch          | 0        |
|    l2_loss        | 0        |
|    l2_norm        | 71.3     |
|    loss           | 0.876    |
|    neglogp        | 0.877    |
|    prob_true_act  | 0.416    |
|    samples_so_far | 32       |
| rollout/          |          |
|    return_max     | 92.7     |
|    return_mean    | 91.3     |
|    return_min     | 90.4     |
|    return_std     | 0.744    |
--------------------------------


...

--------------------------------
| batch_size        | 32       |
| bc/               |          |
|    batch          | 0        |
|    ent_loss       | 0.0017   |
|    entropy        | -1.7     |
|    epoch          | 0        |
|    l2_loss        | 0        |
|    l2_norm        | 91.7     |
|    loss           | -2.18    |
|    neglogp        | -2.18    |
|    prob_true_act  | 8.89     |
|    samples_so_far | 32       |
| rollout/          |          |
|    return_max     | 94.8     |
|    return_mean    | 94       |
|    return_min     | 92.9     |
|    return_std     | 0.788    |
--------------------------------


461batch [00:01, 415.26batch/s]

--------------------------------
| batch_size        | 32       |
| bc/               |          |
|    batch          | 500      |
|    ent_loss       | 0.00219  |
|    entropy        | -2.19    |
|    epoch          | 2        |
|    l2_loss        | 0        |
|    l2_norm        | 93.7     |
|    loss           | -2.68    |
|    neglogp        | -2.68    |
|    prob_true_act  | 14.6     |
|    samples_so_far | 16032    |
| rollout/          |          |
|    return_max     | 94.6     |
|    return_mean    | 94       |
|    return_min     | 91.9     |
|    return_std     | 1.03     |
--------------------------------


676batch [00:01, 359.04batch/s]


Saved DAgger student to: /Users/elisalaegsgaard/Desktop/Speciale/GGSpeciale/code/baseline_code/dagger_models/mountaincar/DAgger_PPO_mountaincar
Skipping unknown algo file: SAC_mountaincar.zip
Skipping unknown algo file: TD3_mountaincar.zip
Skipping unknown algo file: TRPO_mountaincar.zip
  teacher_algo         teacher_file  dagger_timesteps  mean_reward  \
0          PPO  PPO_mountaincar.zip              5000    93.542177   

   std_reward                                      saved_student  
0    1.176743  /Users/elisalaegsgaard/Desktop/Speciale/GGSpec...  

Wrote results to: /Users/elisalaegsgaard/Desktop/Speciale/GGSpeciale/code/baseline_code/dagger_models/mountaincar/dagger_results.csv


## Acrobot

In [24]:
import tempfile
from pathlib import Path
import sys
import numpy as np
import pandas as pd

from imitation.algorithms import bc
from imitation.algorithms.dagger import SimpleDAggerTrainer

from stable_baselines3 import PPO, DDPG, SAC, TD3
from sb3_contrib import TRPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env

# ----------------------------
# Project path handling
# ----------------------------
cwd = Path.cwd()
current = cwd
while not (current / "code" / "baseline_code").exists() and current != current.parent:
    current = current.parent

PROJECT_ROOT = current
BASELINE_CODE = PROJECT_ROOT / "code" / "baseline_code"
sys.path.insert(0, str(BASELINE_CODE))

print("PROJECT_ROOT:", PROJECT_ROOT)

# ----------------------------
# Import Acrobot wrapper
# ----------------------------
from baseline_enviroments.acrobot_env import make_continuous_acrobot

# ----------------------------
# Paths
# ----------------------------
TEACHER_DIR = BASELINE_CODE / "baseline_models" / "acrobot"
DAGGER_OUT_DIR = BASELINE_CODE / "dagger_models" / "acrobot"
DAGGER_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Teacher dir:", TEACHER_DIR)
print("Teacher files:", [p.name for p in TEACHER_DIR.glob("*.zip")])

# ----------------------------
# Algorithm map
# ----------------------------
ALGOS = {
    "PPO": PPO,
    "DDPG": DDPG,
    "SAC": SAC,
    "TD3": TD3,
    "TRPO": TRPO,
}

rng = np.random.default_rng(0)
ENV_FACTORY = make_continuous_acrobot

DAGGER_TIMESTEPS = 50_000
N_EVAL_EPISODES = 10

results = []

teacher_paths = sorted(TEACHER_DIR.glob("*.zip"))
if not teacher_paths:
    raise FileNotFoundError(f"No teacher models found in: {TEACHER_DIR}")

for teacher_zip in teacher_paths:
    algo_name = teacher_zip.name.split("_")[0]

    if algo_name not in ALGOS:
        print(f"Skipping unknown file: {teacher_zip.name}")
        continue

    Algo = ALGOS[algo_name]
    print(f"\n=== DAgger for {algo_name} teacher ===")

    # Create training environment
    venv = make_vec_env(ENV_FACTORY, n_envs=1)

    # Load teacher
    expert = Algo.load(str(teacher_zip), env=None)

    # Student BC trainer
    bc_trainer = bc.BC(
        observation_space=venv.observation_space,
        action_space=venv.action_space,
        rng=rng,
    )

    with tempfile.TemporaryDirectory(prefix=f"dagger_{algo_name}_") as tmpdir:
        dagger_trainer = SimpleDAggerTrainer(
            venv=venv,
            scratch_dir=tmpdir,
            expert_policy=expert,
            bc_trainer=bc_trainer,
            rng=rng,
        )

        dagger_trainer.train(total_timesteps=DAGGER_TIMESTEPS)

        # Evaluate student
        eval_env = make_vec_env(ENV_FACTORY, n_envs=1)
        mean_reward, std_reward = evaluate_policy(
            dagger_trainer.policy,
            eval_env,
            n_eval_episodes=N_EVAL_EPISODES,
        )

        # Save student
        out_path = DAGGER_OUT_DIR / f"DAgger_{algo_name}_acrobot"
        dagger_trainer.policy.save(str(out_path))
        print(f"Saved DAgger student to: {out_path}")

        results.append({
            "teacher_algo": algo_name,
            "mean_reward": mean_reward,
            "std_reward": std_reward,
        })

        eval_env.close()
    venv.close()

df = pd.DataFrame(results).sort_values("mean_reward", ascending=False)
print(df)
df.to_csv(DAGGER_OUT_DIR / "dagger_results.csv", index=False)
print("\nResults saved.")


PROJECT_ROOT: /Users/elisalaegsgaard/Desktop/Speciale/GGSpeciale
Teacher dir: /Users/elisalaegsgaard/Desktop/Speciale/GGSpeciale/code/baseline_code/baseline_models/acrobot
Teacher files: ['PPO_acrobot.zip', 'TRPO_acrobot.zip', 'DDPG_acrobot.zip', 'SAC_acrobot.zip', 'TD3_acrobot.zip']

=== DAgger for DDPG teacher ===


ValueError: <class 'numpy.random._pcg64.PCG64'> is not a known BitGenerator module.